# 02 Geospatial Analysis (Public Version)

This notebook is a clean public version of the spatial part of the project. It uses the synthetic public CSV at `data/sample_anonymized_enrollment.csv` and focuses on distance summaries, grouped origin fields, and public static figures.

No private source records are included here.

## Public Spatial Privacy Note

The public data uses synthetic zones and approximate distance values. The notebook does not perform real geocoding and does not expose record-level map content. All outputs below are aggregate tables or links to public PNG figures.

## Imports

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "sample_anonymized_enrollment.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_cleaning import normalize_columns
from src.geospatial_analysis import (
    distance_bins,
    distance_summary,
    frequency_table,
    public_zone_summary,
)

## Load `data/sample_anonymized_enrollment.csv`

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "sample_anonymized_enrollment.csv"

df = normalize_columns(pd.read_csv(DATA_PATH))

record_summary = pd.DataFrame(
    {
        "metric": ["records", "registry_years", "synthetic_zones"],
        "value": [
            len(df),
            df["registry_year"].nunique(),
            df["synthetic_spatial_zone"].nunique(),
        ],
    }
)
record_summary

## Approximate Distance Analysis

In [ ]:
distance_stats = distance_summary(df, "approx_distance_km").to_frame("value")
distance_stats

## Distance Bins

In [ ]:
distance_band_counts = distance_bins(df, "approx_distance_km")
distance_band_counts

## Guardian/Student Aggregate Categories

In [ ]:
guardian_categories = frequency_table(
    df,
    "guardian_nationality_group",
    top_n=8,
).round(2)

student_categories = frequency_table(
    df,
    "student_nationality_group",
    top_n=8,
).round(2)

occupation_categories = frequency_table(
    df,
    "guardian_occupation_group",
    top_n=8,
).round(2)

display(guardian_categories)
display(student_categories)
display(occupation_categories)

## Synthetic Zone Summary

In [ ]:
zone_stats = public_zone_summary(df, "synthetic_spatial_zone").to_frame("value")
zone_stats

## Spatial Visualization Note

The public site includes static map figures in `assets/screenshots/`. These images are safe portfolio artifacts because they avoid record-level HTML content and are presented as public summaries.

- `assets/screenshots/05_distance_to_school_distribution.png`
- `assets/screenshots/06_guardian_nationality_distribution.png`
- `assets/screenshots/07_student_spatial_density_heatmap.png`
- `assets/screenshots/07_student_spatial_distribution_points.png`

## Key Findings

- The public sample has 180 records spread across 70 synthetic spatial zones.
- Approximate distance has a median near 2.36 km and a mean near 3.15 km.
- Most records fall within 1-2 km in the public sample; 5+ km remains a meaningful outer band.
- Student category counts are highly concentrated in the Argentine group, while guardian categories are more mixed.
- These findings describe the synthetic public sample only.

## Reproducibility Note

Run this notebook from the repository root so `data/sample_anonymized_enrollment.csv` is found. The command-line runner can refresh aggregate text outputs with:

```bash
python src/run_public_analysis.py --input data/sample_anonymized_enrollment.csv --output outputs
```